# Analise Exploratoria 


## AIRPORTS

In [3]:
import pandas as pd

#### Primeiro passo aqui é entender meus dados. O arquivo nao apresenta um header entao eu vou ter que criar 

In [4]:
colunas_airports = [
    "airport_id", "name", "city", "country",
    "iata", "icao", "lat", "lon",
    "altitude", "timezone", "dst", "tz", "type", "source"
]
df_airport = pd.read_csv("airports.dat",header= None,names=colunas_airports)

df_airport.shape


(7698, 14)

In [5]:
df_airport.head()

,airport_id,name,city,country,iata,icao,lat,lon,altitude,timezone,dst,tz,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10,U,Pacific/Port_Moresby,airport,OurAirports


In [6]:
print(f"Valores coluna type: {df_airport["type"].value_counts()}")

Valores coluna type: type
airport    7698
Name: count, dtype: int64


In [7]:
print(f"Valores coluna source: {df_airport["source"].value_counts()}")


Valores coluna source: source
OurAirports    7698
Name: count, dtype: int64


In [8]:
df_airport= df_airport.drop(["type","source"],axis=1)
print(df_airport.head())

   airport_id                                         name          city  \
0           1                               Goroka Airport        Goroka   
1           2                               Madang Airport        Madang   
2           3                 Mount Hagen Kagamuga Airport   Mount Hagen   
3           4                               Nadzab Airport        Nadzab   
4           5  Port Moresby Jacksons International Airport  Port Moresby   

            country iata  icao       lat         lon  altitude timezone dst  \
0  Papua New Guinea  GKA  AYGA -6.081690  145.391998      5282       10   U   
1  Papua New Guinea  MAG  AYMD -5.207080  145.789001        20       10   U   
2  Papua New Guinea  HGU  AYMH -5.826790  144.296005      5388       10   U   
3  Papua New Guinea  LAE  AYNZ -6.569803  146.725977       239       10   U   
4  Papua New Guinea  POM  AYPY -9.443380  147.220001       146       10   U   

                     tz  
0  Pacific/Port_Moresby  
1  Pacific/Port_

### Qualidade dos Dados

In [9]:
#df_airport.isnull().sum()
print(df_airport["iata"].value_counts().head(10))
df_airport["iata"] = df_airport["iata"].replace("\\N", pd.NA)
print(df_airport["iata"].isnull().sum())

iata
\N     1626
GKA       1
MAG       1
HGU       1
LAE       1
POM       1
WWK       1
UAK       1
GOH       1
SFJ       1
Name: count, dtype: int64
1626


#### Como eu uso o IATA como chave no Grafo vou remover os aereoportos sem IATA 

In [10]:
df_airport = df_airport.dropna(subset=["iata"])
print(df_airport.shape)

(6072, 12)


### Distribuição

#### País

In [11]:
pais_count = df_airport["country"].value_counts().head(20)
pais_count

country
United States     1251
Canada             380
Australia          282
China              235
Brazil             210
Russia             177
Indonesia          136
France             124
India              121
United Kingdom     108
Germany             95
Japan               94
Argentina           84
Mexico              77
Iran                69
Colombia            69
Turkey              63
South Africa        59
Norway              56
Italy               56
Name: count, dtype: int64

### Binning da Altitude

In [12]:
df_airport["faixa_altitude"] = pd.cut(df_airport["altitude"], 
                bins=[-100, 0, 500, 1500, 3000, 5000, 15000], right=False,
                labels=["Abaixo do mar", "0-500ft", "500-1500ft", "1500-3000ft", "3000-5000ft", "5000ft+"])

print(df_airport["faixa_altitude"].value_counts().sort_index())

faixa_altitude
Abaixo do mar      12
0-500ft          3442
500-1500ft       1358
1500-3000ft       614
3000-5000ft       387
5000ft+           256
Name: count, dtype: int64


#### Aereoportos abaixo do nivel do mar 

In [13]:
df_airport[df_airport["faixa_altitude"] == "Abaixo do mar"][["name", "city", "country", "altitude"]].sort_values("altitude")

,name,city,country,altitude
4087,Atyrau Airport,Atyrau,Kazakhstan,-72
2069,Ramsar Airport,Ramsar,Iran,-70
2811,Astrakhan Airport,Astrakhan,Russia,-65
4598,Noshahr Airport,Noshahr,Iran,-61
3489,Imperial County Airport,Imperial,United States,-54
3558,El Centro NAF Airport (Vraciu Field),El Centro,United States,-42
2049,Sardar-e-Jangal Airport,Rasht,Iran,-40
5132,Gorgan Airport,Gorgan,Iran,-24
585,Rotterdam The Hague Airport,Rotterdam,Netherlands,-15
583,Lelystad Airport,Lelystad,Netherlands,-13


## Routes

In [14]:
routes = pd.read_csv("routes.dat",header=None)

In [15]:
colunas_routes = [
    "airline", "airline_id", "origin_iata", "origin_id",
    "dest_iata", "dest_id", "codeshare", "stops", "aeronave"
]

df_routes = pd.read_csv("routes.dat", header=None, names=colunas_routes)
df_routes.shape

(67663, 9)

In [16]:
df_routes.isnull().sum()

airline            0
airline_id         0
origin_iata        0
origin_id          0
dest_iata          0
dest_id            0
codeshare      53066
stops              0
aeronave          18
dtype: int64

In [17]:
df_routes = df_routes.drop(columns=["codeshare", "aeronave"])
df_routes.head()

,airline,airline_id,origin_iata,origin_id,dest_iata,dest_id,stops
0,2B,410,AER,2965,KZN,2990,0
1,2B,410,ASF,2966,KZN,2990,0
2,2B,410,ASF,2966,MRV,2962,0
3,2B,410,CEK,2968,KZN,2990,0
4,2B,410,CEK,2968,OVB,4078,0


## Juntando os dois datasets

In [18]:
print(df_routes.dtypes)
print(df_airport.dtypes)

airline          str
airline_id       str
origin_iata      str
origin_id        str
dest_iata        str
dest_id          str
stops          int64
dtype: object
airport_id           int64
name                   str
city                   str
country                str
iata                   str
icao                   str
lat                float64
lon                float64
altitude             int64
timezone               str
dst                    str
tz                     str
faixa_altitude    category
dtype: object


In [19]:
df_routes["origin_id"] = pd.to_numeric(df_routes["origin_id"], errors="coerce")
df_routes["dest_id"] = pd.to_numeric(df_routes["dest_id"], errors="coerce")

In [20]:
# Join com origem
df_merged = df_routes.merge(
    df_airport,
    left_on="origin_id",
    right_on="airport_id",
    suffixes=("", "_origem")
)

# Join com destino
df_merged = df_merged.merge(
    df_airport,
    left_on="dest_id",
    right_on="airport_id",
    suffixes=("_origem", "_destino")
)

df_merged.shape

(66435, 33)

In [22]:
df_merged = df_merged.drop(columns=[
    "iata_origem", "iata_destino",
    "airport_id_origem", "airport_id_destino",
])

In [23]:
df_merged.columns.tolist()

['airline',
 'airline_id',
 'origin_iata',
 'origin_id',
 'dest_iata',
 'dest_id',
 'stops',
 'name_origem',
 'city_origem',
 'country_origem',
 'icao_origem',
 'lat_origem',
 'lon_origem',
 'altitude_origem',
 'timezone_origem',
 'dst_origem',
 'tz_origem',
 'faixa_altitude_origem',
 'name_destino',
 'city_destino',
 'country_destino',
 'icao_destino',
 'lat_destino',
 'lon_destino',
 'altitude_destino',
 'timezone_destino',
 'dst_destino',
 'tz_destino',
 'faixa_altitude_destino']